In [1]:
# 1. Lade das Backend (jetzt ZWEI Dateien!)
%run deduplication.ipynb
from author_cleaning import remove_ghost_papers

def build_cv_pipeline(author_id, end_year=2026):
    print(f"Starte Pipeline für ID: {author_id}...")
    
    # 2. Rohe Daten laden
    raw_data = fetch_openalex_data(author_id)
    df_raw = extract_flat_data(raw_data, author_id)
    if df_raw.empty: return pd.DataFrame()
    
    # 3. PIPELINE SCHRITT A: Autor-Bereinigung (Geister entfernen)
    df_author_clean = remove_ghost_papers(df_raw)
    
    # 4. PIPELINE SCHRITT B: Paper-Deduplizierung (Working Papers entfernen)
    df_clean = deduplicate_papers(df_author_clean)
    
    # 5. Pivot-Tabelle bauen
    df_clean['Inst_Nummer'] = df_clean.groupby('Jahr').cumcount() + 1
    df_clean['Inst_Nummer'] = df_clean['Inst_Nummer'].astype(int)

    df_pivot = df_clean.pivot(index='Jahr', columns='Inst_Nummer', values=['Uni', 'Titel', 'Link'])

    neue_spalten = []
    for col in df_pivot.columns:
        if col[0] == 'Uni': neue_spalten.append(f'Institution_{int(col[1])}')
        elif col[0] == 'Titel': neue_spalten.append(f'Beweis_Paper_{int(col[1])}')
        elif col[0] == 'Link': neue_spalten.append(f'Beweis_Link_{int(col[1])}')
            
    df_pivot.columns = neue_spalten
    df_pivot = df_pivot.reset_index()

    max_inst = int(df_clean['Inst_Nummer'].max()) 
    gewuenschte_reihenfolge = ['Jahr']
    for i in range(1, max_inst + 1):
        if f'Institution_{i}' in df_pivot.columns: gewuenschte_reihenfolge.append(f'Institution_{i}')
        if f'Beweis_Paper_{i}' in df_pivot.columns: gewuenschte_reihenfolge.append(f'Beweis_Paper_{i}')
        if f'Beweis_Link_{i}' in df_pivot.columns: gewuenschte_reihenfolge.append(f'Beweis_Link_{i}')
            
    df_pivot = df_pivot[gewuenschte_reihenfolge]
    start_year = int(df_pivot['Jahr'].min())
    zeitstrahl = pd.DataFrame({'Jahr': range(start_year, end_year + 1)})
    
    return pd.merge(zeitstrahl, df_pivot, on='Jahr', how='left').fillna("-")

# --- AUSFÜHRUNG ---
test_author_id = "A5058372080" # Klaus Schmidt
cv_tabelle_final = build_cv_pipeline(test_author_id)
cv_tabelle_final

Lade und verarbeite Daten...
Starte Pipeline für ID: A5058372080...

--- 👻 GHOST AUTHOR SCAN ---
🌍 Makro-Domäne gesichert: 'Social Sciences'
🔬 Thematischer Kern (Top 3 Fields): ['Economics, Econometrics and Finance', 'Business, Management and Accounting', 'Decision Sciences']
🗑️ Blockiert (Biochemistry, Geneti... | [RAW] Department of ...): 'Genome-wide association analyses of risk...'
🗑️ Blockiert (Social Sciences... | [RAW] Department of ...): 'Chapter 8 The Economics of Fairness, Rec...'
🗑️ Blockiert (Social Sciences... | University of Zurich...): 'Theories of Fairness and Reciprocity: Ev...'
🗑️ Blockiert (Social Sciences... | [RAW] Department of ...): 'The Economics of Fairness, Reciprocity a...'
🗑️ Blockiert (Social Sciences... | [RAW] Department of ...): 'Inequality Aversion, Efficiency, and Max...'
🗑️ Blockiert (Social Sciences... | [RAW] Department of ...): 'Adding a Stick to the Carrot? The Intera...'
🗑️ Blockiert (Engineering... | Illinois State Unive...): 'Fuel Cell Electric

,Jahr,Institution_1,Beweis_Paper_1,Beweis_Link_1,Institution_2,Beweis_Paper_2,Beweis_Link_2,Institution_3,Beweis_Paper_3,Beweis_Link_3,...,Beweis_Link_4,Institution_5,Beweis_Paper_5,Beweis_Link_5,Institution_6,Beweis_Paper_6,Beweis_Link_6,Institution_7,Beweis_Paper_7,Beweis_Link_7
0,1993,University of Bonn,Privatization and Management Incentives in the Transition Period in Eastern Europe,https://api.openalex.org/W2038784833,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
1,1994,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
2,1995,Ifo Institute for Economic Research,Option Contracts and Renegotiation - A Solution to the Hold-Up Problem,https://api.openalex.org/W3123585617,University of Bonn,The interaction of explicit and implicit contracts,https://api.openalex.org/W1572713800,-,-,-,...,-,-,-,-,-,-,-,-,-,-
3,1996,Ludwig-Maximilians-Universität München,Incomplete contracts and privatization,https://api.openalex.org/W1994758860,Institut für Urheber- und Medienrecht,The Costs and Benefits of Privatization: An Incomplete Contracts Approach,https://api.openalex.org/W2174491421,University of Bonn,Reputation in Perturbed Repeated Games,https://api.openalex.org/W1591510394,...,-,-,-,-,-,-,-,-,-,-
4,1997,Ludwig-Maximilians-Universität München,Debt as an Option to Own in the Theory of Ownership Rights,https://api.openalex.org/W972896655,Ludwig-Maximilians-Universität München,"Methods of Privatization: Auctions, Bargaining, and Give-Aways",https://api.openalex.org/W3126026928,Institut für Urheber- und Medienrecht,Managerial Incentives and Product Market Competition,https://api.openalex.org/W2110821787,...,-,-,-,-,-,-,-,-,-,-
5,1998,Ludwig-Maximilians-Universität München,Sequential Investments and Options to Own,https://api.openalex.org/W2030584975,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
6,1999,Institut für Urheber- und Medienrecht,"A Theory of Fairness, Competition, and Cooperation",https://api.openalex.org/W3022808291,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
7,2000,Ludwig-Maximilians-Universität München,The political economy of mass privatization and the risk of expropriation,https://api.openalex.org/W1836841837,Ludwig-Maximilians-Universität München,"Fairness, incentives, and contractual choices",https://api.openalex.org/W2122587608,-,-,-,...,-,-,-,-,-,-,-,-,-,-
8,2001,"[RAW] University of Munich CESifo, and CEPR",Convertible Securities and Venture Capital Finance,https://api.openalex.org/W1485831114,Ludwig-Maximilians-Universität München,"Fairness, Incentives and Contractual Incompleteness",https://api.openalex.org/W1930580325,Ludwig-Maximilians-Universität München,Theories of fairness and reciprocity - evidence and economic applications,https://api.openalex.org/W1839516636,...,-,-,-,-,-,-,-,-,-,-
9,2002,"[RAW] 2Universität München, CESifo and CEPR","Der Markt für Venture Capital: Anreizprobleme, Governance Strukturen und staatliche Interventionen",https://api.openalex.org/W2129485090,[RAW] University of Munichklaus.schmidt@lrz.uni–muenchen.de,Discrete-Time Approximations of the Holmstrom-Milgrom Brownian-Motion Model of Intertemporal Incentive Provision,https://api.openalex.org/W2155364760,-,-,-,...,-,-,-,-,-,-,-,-,-,-
